# 图文相似度分析 Demo

**目标**: 分析文本描述与图像内容的一致性

**模型**:
- 视觉理解: Qwen2.5-VL-7B-Instruct
- 文本向量化: BGE-M3
- 中文分词: jieba

## 1. 检查环境

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. 安装依赖（如需要）

In [ ]:
# 首次运行请取消注释执行
# !pip install -r requirements.txt

## 3. 配置环境变量

In [ ]:
import os

# 使用 HuggingFace 国内镜像（可选）
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# 模型缓存目录
CACHE_DIR = "./models"
os.makedirs(CACHE_DIR, exist_ok=True)

## 4. 准备输入数据

In [ ]:
# 单条数据测试
input_data = {
    "id": "001",
    "text": "酒店位于市中心，装修豪华",
    "image_path": "https://p3-sign.xhscdn.com/abc/xxx.webp"  # 替换为实际URL
}

# 或使用本地图片
# input_data = {
#     "id": "001",
#     "text": "酒店位于市中心，装修豪华",
#     "image_path": "/path/to/image.jpg"
# }

print("输入数据:", input_data)

## 5. 加载模型

In [ ]:
import sys
sys.path.insert(0, '.')

from src.image_processor import ImageProcessor
from src.text_processor import TextProcessor
from src.vectorizer import Vectorizer
from src.similarity import SimilarityCalculator
from src.pipeline import ImageTextPipeline

In [ ]:
# 创建 Pipeline
pipeline = ImageTextPipeline(
    provider="huggingface",  # 使用 HuggingFace 模型
    cache_dir=CACHE_DIR
)

print("模型加载中，请稍候...")

## 6. 执行分析

In [ ]:
import time

start_time = time.time()

# 执行分析
result = pipeline.analyze(
    text=input_data["text"],
    image_path=input_data["image_path"]
)

elapsed_time = time.time() - start_time
result["processing_time"] = elapsed_time

print(f"\n⏱️  处理时间: {elapsed_time:.2f} 秒")

## 7. 查看结果

In [ ]:
import json

print("="*50)
print("📊 图文相似度分析结果")
print("="*50)

print(f"\n🎯 综合相似度: {result.get('average_similarity', 0):.4f}")
print(f"   名词相似度: {result.get('noun_similarity', 0):.4f}")
print(f"   形容词相似度: {result.get('adjective_similarity', 0):.4f}")

print(f"\n📝 文本关键词:")
print(f"   名词: {result.get('text_nouns', [])}")
print(f"   形容词: {result.get('text_adjectives', [])}")

print(f"\n🖼️ 图像关键词:")
print(f"   名词: {result.get('image_nouns', [])}")
print(f"   形容词: {result.get('image_adjectives', [])}")

print(f"\n📄 原始响应:")
print(f"   {result.get('raw_image_response', '')[:200]}...")

print("\n" + "="*50)
print("💡 解读: 相似度越高，说明文本描述与图像内容越一致")
print("="*50)

## 8. 清理显存（如需处理多张图片）

In [ ]:
# 处理完一张图片后清理显存
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"显存已清理，当前使用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")